In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/bank-full.csv", sep=";")
num_cols = df.select_dtypes(include="int").columns
cat_cols = df.select_dtypes(include="str").columns
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

This dataset does not have missing values, duplicates and columns are either str or int. Number of rows is 45211 and number of columns is 17. Target name - y, tells us has the client subscribed a term deposit after campaign calls?

In [ ]:
df.describe()

Few things that caught my eye are negative balance i think it's possible but i will research this anyway, outliers especially in columns pdays, previous (also this two can correlate which can harm our model in future), duration looks strange. From data dictionary -1 in pdays means client was not previously contacted)

In [ ]:
for col in df.select_dtypes(include="str").columns:
    print("Column: ", end="")
    print(df[col].value_counts())

In [ ]:
df["y"].value_counts(normalize=True)

First of all target is very imbalanced 0.88 to 0.12 i need to be cautious with this one cause model who just always predict no will have accuracy of 0.88. I will use either precision or recall and instead of ROC-AUC i will use PR-AUC that will be more fair. After that poutcome is interesting column and can tell more to the model however 37k is unknown. Also default column is interesting i think if bank will provide them loan this people with chance will accept it because other banks reject them. Other columns seems to be normal but with imputed category "Unknown"

In [ ]:
for col in num_cols:
    g = sns.histplot(df, x=col)
    g.figure.suptitle(f"{col} distribution")
    plt.show()

Every column beside age, day are highly right skewed.

In [ ]:
sns.histplot(df["balance"], log_scale=True)
plt.show()

Log of the balance look a little better but still exclude negative values. I will use some sort of transfromation on this column rather than default balance because of it skewness. I want to create bins and understand what is that peak is this exactly zero or other number.

In [ ]:
df[df["balance"] == 0].shape

In [ ]:
balance_bins = [-9000, 0, 1, 5000, 20000, np.inf]
balance_labels = ["negative", "zero", "low", "medium", "high"]
balance_df = df.copy()
balance_df["balance_size"] = pd.cut(
    balance_df["balance"], bins=balance_bins, labels=balance_labels, right=False
)
balance_df["balance_size"].value_counts()

Actually majority of balance are not zero or negative but is just a low amount of money. Now i want to see event rate for each numeric column in order to understand shape of the relationship between numeric columns and target. Thus understand what to do with each column.

In [ ]:
y_num = (df["y"] == "yes").astype(int)

for col in num_cols:
    bins = pd.qcut(df[col], 10, duplicates="drop")
    rates = y_num.groupby(bins, observed=True).mean()
    print(f"{col} : {rates}")

Many interesting things i found. 1)Age relationship is not linear and look like U-shaped rel cause we have young people with high rate then middle-aged people with lower and then seniors with higher rate. 2)Balance here we see strong linear relationship higher balance = higher rate. 3) Day is not important and i will drop it probably. 4) The most interesting Duration what we see here very strong linear relationship longer duration = higher rate but there are two problems. To start with we are trying to predict who we need to call but information about duration of the call we will get only after the call. In addition, low duration = instant no but higher durations = customer is interested = yes, but actually it is reversed interested customer = yes and not interested = no. 5) Campaign or number of call during it. what we see here more call = less yes , it is because people can become annoyed and if people are interested they will accept offer in first calls. 6) pdays and previous are highly skewed and is hard to tell something.


I looked at the data dictionary and my thoughts about duration was right and author suggest to drop this column

```
last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
```

Okay now lets check the same thing but for the categorical columns

In [ ]:
df_groups = df.copy()
df_groups["y_num"] = (df_groups["y"] == "yes").astype(int)
for col in cat_cols:
    if col == "y":
        continue
    grouped_df = df_groups.groupby(col).agg(
        count=("y_num", "count"), mean_rate=("y_num", "mean")
    )
    print(grouped_df.sort_values(by="mean_rate", ascending=False))

After this we can understand that highest rate of subscription has student and retired people. However, blue-collars,house-maid and services has lowest rate. Single person tend to subscribe more often than married or divorced people. We can see that education has some effect and people with higher education have higher rate. Also very important are default, housing and loan. Looking at the numbers we see that people with previous default have a very low rate of subscriptions because they are do not have money to invest. Same thing with the loan and housing they already spend money to close this loans. On the other hand we have month with some rate at 50% it looks good but can be pure luck. I dont think that people tend to subscribe more in march and lowest in may. However we can see that between this 2 months are 13k of observations and i think more observations = lower rate because in gereral we can see that only 12% rate of acceptance. In the end we have poutcome very strong predictor 64% rate at previous success and 12% at previous failure.

Also i want to check some things about Unknown values. We can see that job and education have Unknown category but also poutcome and there 2 different stories. I suggest unknown in job/education means customer just did not specified. But poutcome's unknown means that customer did not participated in last campaign. In addition to poutcome we have pdays which we explored early where -1 means was not previously contacted. I want to check if these two are the same observations.

In [ ]:
print(df[df["pdays"] == -1].shape)
print(df[df["poutcome"] == "unknown"].shape)
print(df[(df["poutcome"] == "unknown") & (df["pdays"] == -1)].shape)

Okay 36954 observations have both poutcome = unknown and pdays = -1. There are 5 observations where pdays = -1 but poutcome is not unknown i want to look at them and understand what is wrong with them.

In [ ]:
print(df[(df["poutcome"] != "unknown") & (df["pdays"] == -1)].shape)

There is no such observations it's strange i want to check crosstab

In [ ]:
pd.crosstab(df["pdays"] == -1, df["poutcome"])

In [ ]:
print(df[(df["poutcome"] == "unknown") & (df["pdays"] != -1)].head())

Okay now we see that some sort of imputation error cause previous calls was 100% but poutcome is unknown. It will be strange to these 5 observations if bussines just didn't get result. I will delete them cause 5 observations from 45k is nothing and won't affect our model much.

In [ ]:
df_heatmap = df.copy()
df_heatmap["y_num"] = (df_heatmap["y"] == "yes").astype(int)

methods = ["spearman", "pearson"]

for method in methods:
    matrix = df_heatmap.corr(method=method, numeric_only=True)
    g = sns.heatmap(matrix, annot=True, fmt=".2f")
    g.figure.suptitle(f"{method} correlation")
    plt.show()

We can that pdays and previous are highly correlated, so only one of them should remain.

## Feature verdicts

| Feature | Verdict | Why |
|---|---|---|
| `age` | bin into ranges → OHE | relationship is **U-shaped** (young and senior high, middle low); a linear model can't use raw age |
| `job` | OHE | rates differ enough across jobs (student/retired ~0.23–0.29 vs blue-collar 0.07) |
| `marital` | OHE | weaker than job, but a real difference exists (single 0.15 > married 0.10) |
| `education` | **OHE** (not Ordinal) | order exists (primary < secondary < tertiary), **but** `unknown` (4.1%) has no rank and its rate (0.136) sits *between* secondary and tertiary — OrdinalEncoder would impose a false order, so OHE is safer |
| `default` | OHE | modest, rare signal (`yes` = 1.8% of rows, rate 0.064 vs `no` 0.118). Note: my earlier hypothesis was **wrong** — `default=yes` subscribe *less*, not more (no spare money to invest) |
| `balance` | `PowerTransformer("yeo-johnson")` | strongly right-skewed **with negative values**, so `log`/`log1p` fails (NaN); yeo-johnson handles negatives |
| `housing` | OHE | clear signal (`no` 0.167 vs `yes` 0.077) |
| `loan` | OHE | signal present (`no` 0.127 vs `yes` 0.067) |
| `contact` | OHE | strong signal (`unknown` 0.04 vs `cellular` 0.15) |
| `day` | **drop** | flat / noisy event rate, ρ ≈ −0.03; no real relationship, adds noise |
| `month` | OHE | known **before** the call and shows spread, so keep — **but** 3 years (2008–2010) are collapsed into 12 labels, so it's confounded with year, not clean seasonality |
| `duration` | **drop** | leakage: known only *after* the call, reverse causality (interested client → long call), no field in the API contract |
| `campaign` | keep | monotonic: more contacts → lower rate |
| `pdays` | **drop** | ρ = 0.99 with `previous` (shared `-1` sentinel) → redundant; keep `previous` instead |
| `previous` | keep | signal present; count of prior contacts |
| `poutcome` | OHE | strongest categorical (`success` 0.65 vs `unknown` 0.09); `unknown` = "no prior campaign" = information, keep as its own column |
| `y` | → int target | encode `yes` / `no` as 1 / 0 |

**No blanket outlier removal.** The large values in `balance`, `campaign`, `previous` are **real, not errors**. Removing them would bias the model and, at 11.7% positives, delete minority-class rows. Skew is handled by the transforms above (yeo-johnson); trees are insensitive to outliers anyway. The only row deletion is the 5 inconsistent `poutcome=unknown` / `pdays≠-1`
